# ViNewsQA — Qwen2.5-1.5B LoRA-family fine-tuning

Pipeline 17 cells for SQuAD 1.1 extractive QA:
1. Train **LoRA**, **TinyLoRA**, **DoRA**, and **DeLoRA**.
2. Use dev for early stopping and evaluate on the full 1,987-example test split.
3. Compare EM/F1 (maximum over all gold answers).
4. Optionally export per-adapter predictions.

Run the installation cells, restart the kernel, then run all remaining cells. The train cell must print
`>>> TRAIN_FIX_V7_PICKLE <<<` and `>>> pickle-fix <<<` before the first checkpoint save.


In [ ]:
!pip uninstall torch torchvision torchaudio xformers transformers trl unsloth unsloth_zoo peft -y


In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128 --no-cache-dir


In [ ]:
# Install the stack first, then pin PEFT LAST so Unsloth cannot replace the required API.
!pip install transformers trl accelerate bitsandbytes xformers datasets safetensors scikit-learn --no-cache-dir
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" unsloth_zoo --no-cache-dir
!pip install "peft==0.19.1" --no-deps --force-reinstall --no-cache-dir


In [ ]:
import importlib
import inspect

for pkg in ["torch", "transformers", "datasets", "trl", "unsloth", "peft"]:
    importlib.import_module(pkg)
    print(f"OK  {pkg}")

import peft
from peft import TinyLoraConfig, DeloraConfig

if peft.__version__ != "0.19.1":
    raise RuntimeError(f"Expected peft==0.19.1, got {peft.__version__}; restart the kernel.")
tiny_sig = inspect.signature(TinyLoraConfig.__init__).parameters
delora_sig = inspect.signature(DeloraConfig.__init__).parameters
if "r" not in tiny_sig or not {"r", "delora_lambda", "target_modules", "module_dropout"} <= set(delora_sig):
    raise RuntimeError("PEFT TinyLoraConfig/DeloraConfig API is incomplete.")
print("TinyLoraConfig OK | DeloraConfig OK | PEFT", peft.__version__)


In [ ]:
import logging
import os
import warnings

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTHONWARNINGS"] = "ignore"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["ACCELERATE_BYPASS_DEVICE_MAP"] = "true"
os.environ["ACCELERATE_NUM_PROCESSES"] = "1"
for name in ("transformers", "datasets", "torch", "unsloth", "peft", "accelerate"):
    logging.getLogger(name).setLevel(logging.ERROR)
print("Training environment configured.")


In [ ]:
import gc
import json
import math
import re
import string
import unicodedata
from collections import Counter
from pathlib import Path

import torch
from datasets import Dataset
from tqdm import tqdm
from transformers import AutoTokenizer

NOTEBOOK_VERSION = "TRAIN_FIX_V7_PICKLE"
BASE_MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"
DATASET_ROOT = Path("../../")
TRAIN_JSON_PATH = DATASET_ROOT / "train_ViNewsQA.json"
DEV_JSON_PATH = DATASET_ROOT / "dev_ViNewsQA.json"
TEST_JSON_PATH = DATASET_ROOT / "test_ViNewsQA.json"
PROFILING_CONFIG_PATH = "profiling_config_vinewsqa.json"
COMPARE_EVAL_PATH = "eval_compare_adapters_vinewsqa_test.json"

SYSTEM_PROMPT = (
    "Bạn là hệ thống hỏi-đáp trích xuất tiếng Việt. Chỉ trả lời bằng một cụm từ xuất hiện "
    "nguyên văn trong đoạn văn, không giải thích và không thêm tiền tố.\n\n"
    "Đoạn văn:\n{context}"
)

RUN_TRAINING = True
RUN_METRIC_EVAL = True
RUN_SUBMISSION_EXPORT = False
LOAD_IN_4BIT = True
LOAD_IN_8BIT = False
MAX_SEQ_CAP = 4096
MIN_SEQ_LENGTH = 512
MAX_NEW_TOKENS = 64
MIN_FREE_VRAM_GIB = 8.0
INFER_BATCH_SIZE = 16

EVAL_SPLIT = "test"
EXPECTED_TEST_SIZE = 1987
EVAL_MAX_SAMPLES = None
SUBMISSION_MAX_SAMPLES = None
EVAL_LOG_EVERY = 100
SUBMISSION_LOG_EVERY = 100

TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
USE_EARLY_STOPPING = True
MAX_TRAIN_EPOCHS = 5
EARLY_STOPPING_PATIENCE = 3
EARLY_STOPPING_THRESHOLD = 0.001
EVAL_STEPS = 200
SAVE_STEPS = 200
SAVE_TOTAL_LIMIT = 3

TRAIN_COMMON = dict(
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_ratio=0.03,
    num_train_epochs=MAX_TRAIN_EPOCHS,
    learning_rate=2e-4,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
)

ADAPTER_VARIANTS = [
    {"name": "lora", "save_path": "qwen2.5-1.5b-instruct-lora-vinewsqa", "output_dir": "outputs_vinewsqa_lora"},
    {"name": "tinylora", "save_path": "qwen2.5-1.5b-instruct-tinylora-vinewsqa", "output_dir": "outputs_vinewsqa_tinylora"},
    {"name": "dora", "save_path": "qwen2.5-1.5b-instruct-dora-vinewsqa", "output_dir": "outputs_vinewsqa_dora"},
    {"name": "delora", "save_path": "qwen2.5-1.5b-instruct-delora-vinewsqa", "output_dir": "outputs_vinewsqa_delora"},
]
TRAIN_METHODS = ["lora", "tinylora", "dora", "delora"]
EVAL_METHODS = ["lora", "tinylora", "dora", "delora"]

# Giống ViCoQA: thanh tiến độ train dạng n/total.
TQDM_BAR = "{desc}: {percentage:3.0f}%|{bar:30}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"


def load_in_4bit_for_method(method_name):
    return False if method_name == "delora" else LOAD_IN_4BIT


def build_messages(sample, for_inference=False):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT.format(context=sample["context"])},
        {"role": "user", "content": sample["question"]},
    ]
    if not for_inference:
        messages.append({"role": "assistant", "content": sample["answer"]})
    return messages


def sample_to_train_text(sample, tokenizer):
    return tokenizer.apply_chat_template(build_messages(sample), tokenize=False, add_generation_prompt=False)


def load_tokenizer(model_path=BASE_MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    if tokenizer.chat_template is None:
        raise RuntimeError(f"Tokenizer {model_path} has no chat template.")
    return tokenizer


def clear_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()


print(">>> TRAIN_FIX_V7_PICKLE <<<")
print("PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())


In [ ]:
def load_squad11_split(path):
    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)
    if payload.get("version") != "1.1" or not isinstance(payload.get("data"), list):
        raise ValueError(f"{path} is not a SQuAD 1.1 object.")

    samples = []
    dropped = 0
    for article in payload["data"]:
        title = str(article.get("title") or "")
        for paragraph in article.get("paragraphs") or []:
            context = str(paragraph.get("context") or "").strip()
            for qa in paragraph.get("qas") or []:
                question = str(qa.get("question") or "").strip()
                gold_answers = []
                for answer in qa.get("answers") or []:
                    text = str(answer.get("text") or "").strip()
                    if text and text not in gold_answers:
                        gold_answers.append(text)
                if not context or not question or not gold_answers:
                    dropped += 1
                    continue
                samples.append({
                    "id": str(qa.get("id") or ""),
                    "title": title,
                    "context": context,
                    "question": question,
                    "answer": gold_answers[0],
                    "gold_answers": gold_answers,
                })
    if not samples:
        raise RuntimeError(f"No valid labeled samples in {path}.")
    ids = [row["id"] for row in samples]
    if any(not value for value in ids) or len(ids) != len(set(ids)):
        raise RuntimeError(f"{path} contains empty or duplicate QA ids.")
    print(f"{path.name}: {len(samples)} samples | dropped={dropped}")
    return samples


train_samples = load_squad11_split(TRAIN_JSON_PATH)
dev_samples = load_squad11_split(DEV_JSON_PATH)
test_samples = load_squad11_split(TEST_JSON_PATH)
if len(test_samples) != EXPECTED_TEST_SIZE:
    raise RuntimeError(f"Expected {EXPECTED_TEST_SIZE} test samples, got {len(test_samples)}.")

if EVAL_SPLIT == "test":
    eval_samples = test_samples
elif EVAL_SPLIT == "dev":
    eval_samples = dev_samples
else:
    raise ValueError(f"Unsupported EVAL_SPLIT={EVAL_SPLIT!r}")

print(f"Early stopping split: dev ({len(dev_samples)})")
print(f"Metric evaluation split: {EVAL_SPLIT} ({len(eval_samples)})")


In [ ]:
def compute_max_seq_length(samples, tokenizer, cap=MAX_SEQ_CAP, min_len=MIN_SEQ_LENGTH):
    lengths = [
        len(tokenizer.encode(sample_to_train_text(sample, tokenizer)))
        for sample in tqdm(samples, desc="Token profiling", unit="sample", bar_format=TQDM_BAR)
    ]
    lengths.sort()
    n = len(lengths)
    stats = {
        "min": lengths[0],
        "p50": lengths[n // 2],
        "p95": lengths[min(n - 1, int(n * 0.95))],
        "p99": lengths[min(n - 1, int(n * 0.99))],
        "max": lengths[-1],
    }
    chosen = max(min(math.ceil(stats["p99"] * 1.05), cap), min_len)
    chosen = min(cap, ((chosen + 255) // 256) * 256)
    stats["chosen_max_seq_length"] = chosen
    stats["truncated_samples"] = sum(length > chosen for length in lengths)
    stats["truncated_pct"] = round(100 * stats["truncated_samples"] / n, 3)
    return chosen, stats


tokenizer_prof = load_tokenizer()
max_seq_length, length_stats = compute_max_seq_length(train_samples, tokenizer_prof)
with open(PROFILING_CONFIG_PATH, "w", encoding="utf-8") as handle:
    json.dump({"max_seq_length": max_seq_length, "token_length_stats": length_stats}, handle, indent=2)
print("max_seq_length =", max_seq_length)
print(length_stats)
del tokenizer_prof
clear_gpu()


In [ ]:
tokenizer_fmt = load_tokenizer()


def formatting_prompts_func(examples):
    texts = []
    for context, question, answer in zip(examples["context"], examples["question"], examples["answer"]):
        sample = {"context": context, "question": question, "answer": answer}
        texts.append(tokenizer_fmt.apply_chat_template(build_messages(sample), tokenize=False, add_generation_prompt=False))
    return {"text": texts}


train_hf = Dataset.from_list(train_samples)
dataset = train_hf.map(formatting_prompts_func, batched=True, remove_columns=train_hf.column_names)
dev_hf = Dataset.from_list(dev_samples)
eval_dataset = dev_hf.map(formatting_prompts_func, batched=True, remove_columns=dev_hf.column_names)
print(f"Train={len(dataset)} | early-stop dev={len(eval_dataset)}")
print(dataset[0]["text"][:500])


## Train four adapter variants

LoRA, TinyLoRA, and DoRA load the base model in 4-bit. DeLoRA loads full BF16/FP16 weights because
its initialization computes norms over the original floating-point weights.


In [ ]:
def apply_adapter(model, method_name):
    from unsloth import FastLanguageModel

    def resolve_causallm(value):
        if hasattr(value, "prepare_inputs_for_generation"):
            return value
        if hasattr(value, "get_base_model") and hasattr(value.get_base_model(), "prepare_inputs_for_generation"):
            return value.get_base_model()
        raise RuntimeError("Could not resolve a causal-LM wrapper.")

    if method_name in {"lora", "dora"}:
        model = FastLanguageModel.get_peft_model(
            model,
            r=16,
            lora_alpha=32,
            target_modules=TARGET_MODULES,
            lora_dropout=0.05,
            bias="none",
            use_gradient_checkpointing="unsloth",
            random_state=3407,
            use_dora=(method_name == "dora"),
        )
        print(f"Applied {method_name}: r=16, alpha=32")
        return model

    if method_name == "tinylora":
        import inspect
        from peft import TinyLoraConfig, get_peft_model

        parameters = inspect.signature(TinyLoraConfig.__init__).parameters
        desired = {
            "r": 2, "u": 64, "num_projections": 64, "target_modules": TARGET_MODULES,
            "tinylora_dropout": 0.0, "lora_dropout": 0.0, "bias": "none",
            "task_type": "CAUSAL_LM", "weight_tying": 0.0, "projection_seed": 3407,
            "init_weights": True,
        }
        kwargs = {key: value for key, value in desired.items() if key in parameters}
        model = get_peft_model(resolve_causallm(model), TinyLoraConfig(**kwargs))
    elif method_name == "delora":
        from peft import DeloraConfig, get_peft_model

        config = DeloraConfig(
            r=16,
            delora_lambda=15,
            target_modules=TARGET_MODULES,
            module_dropout=0.05,
            bias="none",
            task_type="CAUSAL_LM",
            init_weights=True,
        )
        model = get_peft_model(resolve_causallm(model), config)
    else:
        raise ValueError(f"Unknown method: {method_name}")

    model.config.use_cache = False
    if hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
    model.print_trainable_parameters()
    return model


def train_one_variant(variant, max_seq_length, dataset, eval_dataset=None):
    import inspect
    from unsloth import FastLanguageModel, is_bfloat16_supported
    from trl import SFTTrainer
    try:
        from trl import SFTConfig
    except ImportError:
        SFTConfig = None
    from transformers import EarlyStoppingCallback, TrainerCallback, TrainingArguments

    class TrainProgressCallback(TrainerCallback):
        """Ép hiện tiến độ step/total giống ViCoQA khi Unsloth ẩn thanh HF mặc định."""

        def __init__(self):
            self.pbar = None

        def on_train_begin(self, args, state, control, **kwargs):
            total = int(state.max_steps or 0)
            if total <= 0:
                return
            self.pbar = tqdm(
                total=total,
                desc="Train",
                unit="step",
                bar_format=TQDM_BAR,
                dynamic_ncols=True,
                mininterval=0.5,
            )
            print(f"[Train] progress bar ON | 0/{total} steps", flush=True)

        def on_step_end(self, args, state, control, **kwargs):
            if self.pbar is None:
                return
            self.pbar.n = int(state.global_step)
            self.pbar.refresh()

        def on_log(self, args, state, control, logs=None, **kwargs):
            if self.pbar is None or not logs:
                return
            postfix = {}
            for key in ("loss", "eval_loss", "learning_rate", "epoch"):
                if key in logs and logs[key] is not None:
                    val = logs[key]
                    postfix[key] = f"{val:.4f}" if isinstance(val, float) else val
            if postfix:
                self.pbar.set_postfix(postfix, refresh=True)

        def on_train_end(self, args, state, control, **kwargs):
            if self.pbar is not None:
                self.pbar.n = int(state.global_step or self.pbar.total or 0)
                self.pbar.refresh()
                self.pbar.close()
                self.pbar = None

    print(">>> TRAIN_FIX_V7_PICKLE <<<", flush=True)
    name = variant["name"]
    use_4bit = load_in_4bit_for_method(name)
    if not torch.cuda.is_available():
        raise RuntimeError("Qwen2.5-1.5B training requires a CUDA GPU.")
    clear_gpu()
    print(f"TRAIN {name.upper()} | load_in_4bit={use_4bit}", flush=True)

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME,
        max_seq_length=max_seq_length,
        dtype=None,
        load_in_4bit=use_4bit,
        load_in_8bit=LOAD_IN_8BIT,
    )
    model = apply_adapter(model, name)
    eval_on = USE_EARLY_STOPPING and eval_dataset is not None

    common_args = dict(
        **TRAIN_COMMON,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        output_dir=variant["output_dir"],
        disable_tqdm=False,
        logging_strategy="steps",
        logging_steps=SAVE_STEPS,
        logging_first_step=False,
        log_level="error",
        log_level_replica="error",
        save_strategy="steps",
        save_steps=SAVE_STEPS,
        save_total_limit=SAVE_TOTAL_LIMIT,
        report_to="none",
        dataloader_num_workers=0,
        save_safetensors=True,
    )
    callbacks = [TrainProgressCallback()]
    args_cls = SFTConfig if SFTConfig is not None else TrainingArguments
    args_params = inspect.signature(args_cls.__init__).parameters
    eval_key = "eval_strategy" if "eval_strategy" in args_params else "evaluation_strategy"
    if eval_on:
        common_args.update({
            eval_key: "steps",
            "eval_steps": EVAL_STEPS,
            "load_best_model_at_end": True,
            "metric_for_best_model": "eval_loss",
            "greater_is_better": False,
        })
        callbacks.append(EarlyStoppingCallback(
            early_stopping_patience=EARLY_STOPPING_PATIENCE,
            early_stopping_threshold=EARLY_STOPPING_THRESHOLD,
        ))
    else:
        common_args[eval_key] = "no"

    # Prefer SFTConfig and place SFT-only dataset options there when its installed API supports them.
    if SFTConfig is not None:
        for key, value in (
            ("max_seq_length", max_seq_length),
            ("dataset_text_field", "text"),
            ("packing", False),
            ("dataset_num_proc", 1),
        ):
            if key in args_params:
                common_args[key] = value
    filtered_args = {key: value for key, value in common_args.items() if key in args_params or key == "output_dir"}
    training_config = args_cls(**filtered_args)

    trainer_kwargs = dict(
        model=model,
        train_dataset=dataset,
        eval_dataset=eval_dataset if eval_on else None,
        args=training_config,
        callbacks=callbacks,
    )
    trainer_params = inspect.signature(SFTTrainer.__init__).parameters
    if "processing_class" in trainer_params:
        trainer_kwargs["processing_class"] = tokenizer
    elif "tokenizer" in trainer_params:
        trainer_kwargs["tokenizer"] = tokenizer
    if SFTConfig is None:
        for key, value in (
            ("max_seq_length", max_seq_length),
            ("dataset_text_field", "text"),
            ("packing", False),
            ("dataset_num_proc", 1),
        ):
            if key in trainer_params:
                trainer_kwargs[key] = value

    trainer = SFTTrainer(**trainer_kwargs)
    # Unsloth đôi khi ghi đè disable_tqdm — ép lại như ViCoQA.
    trainer.args.disable_tqdm = False

    print(">>> pickle-fix <<<", flush=True)
    import os
    import sys
    import json
    import trl
    import trl.trainer.sft_config as sft_config_mod
    import trl.trainer.sft_trainer as sft_trainer_mod
    from transformers.trainer import Trainer as HFTrainer

    cfg_cls, tr_cls = type(trainer.args), type(trainer)
    # Unsloth dùng UnslothSFTConfig nhưng pickle vẫn resolve tên SFTConfig → đăng ký cả hai.
    cfg_cls.__module__ = "trl.trainer.sft_config"
    tr_cls.__module__ = "trl.trainer.sft_trainer"
    for attr in ("SFTConfig", "UnslothSFTConfig"):
        setattr(sft_config_mod, attr, cfg_cls)
        if hasattr(trl, attr):
            setattr(trl, attr, cfg_cls)
    for attr in ("SFTTrainer", "UnslothSFTTrainer"):
        setattr(sft_trainer_mod, attr, tr_cls)
        if hasattr(trl, attr):
            setattr(trl, attr, tr_cls)
    sys.modules["trl.trainer.sft_config"] = sft_config_mod
    sys.modules["trl.trainer.sft_trainer"] = sft_trainer_mod

    def _is_training_args_obj(obj):
        name = type(obj).__name__
        return (
            "SFTConfig" in name
            or "TrainingArguments" in name
            or name in {"SFTConfig", "UnslothSFTConfig", "TrainingArguments", "UnslothTrainingArguments"}
        )

    def _write_args_json(obj, path_hint):
        json_path = str(path_hint).replace(".bin", ".json")
        if not json_path.endswith(".json"):
            json_path = os.path.join(str(path_hint), "training_args.json") if os.path.isdir(str(path_hint)) else str(path_hint) + ".json"
        os.makedirs(os.path.dirname(json_path) or ".", exist_ok=True)
        if hasattr(obj, "to_json_string"):
            with open(json_path, "w", encoding="utf-8") as jf:
                jf.write(obj.to_json_string())
        else:
            payload = obj.to_dict() if hasattr(obj, "to_dict") else {"repr": repr(obj)}
            with open(json_path, "w", encoding="utf-8") as jf:
                json.dump(payload, jf, ensure_ascii=False, indent=2, default=str)
        print(f"[pickle-fix] wrote {json_path} ({type(obj).__name__})", flush=True)
        return json_path

    # KHÔNG pickle UnslothSFTConfig: ghi JSON trước khi torch.save kịp crash.
    _orig_torch_save = torch.save

    def _torch_save_safe(obj, f, *args, **kwargs):
        path = getattr(f, "name", None) or str(f)
        if _is_training_args_obj(obj) or "training_args" in str(path):
            _write_args_json(obj, path)
            return
        return _orig_torch_save(obj, f, *args, **kwargs)

    torch.save = _torch_save_safe
    torch._vinewsqa_pickle_patch = True

    def _install_save_wrapper(cls):
        # Luôn cài lại wrapper (Unsloth có thể thay class sau mỗi from_pretrained).
        _orig_save = cls._save

        def _save_fixed(self, output_dir=None, state_dict=None):
            out = output_dir or self.args.output_dir
            os.makedirs(out, exist_ok=True)
            model_to_save = self.model
            if hasattr(self, "accelerator"):
                try:
                    model_to_save = self.accelerator.unwrap_model(self.model)
                except Exception:
                    model_to_save = self.model
            if state_dict is None:
                model_to_save.save_pretrained(out, safe_serialization=True)
            else:
                model_to_save.save_pretrained(out, state_dict=state_dict, safe_serialization=True)
            tok = getattr(self, "processing_class", None) or getattr(self, "tokenizer", None)
            if tok is not None and hasattr(tok, "save_pretrained"):
                tok.save_pretrained(out)
            _write_args_json(self.args, os.path.join(out, "training_args.bin"))

        cls._save = _save_fixed
        cls._vinewsqa_save_patched = True

    _install_save_wrapper(HFTrainer)
    _install_save_wrapper(tr_cls)
    for base in getattr(tr_cls, "__mro__", ()):
        if base is object:
            continue
        if hasattr(base, "_save"):
            _install_save_wrapper(base)
    print(
        f"[pickle-fix] V7 ready | args={cfg_cls.__module__}.{cfg_cls.__name__} | trainer={tr_cls.__name__}",
        flush=True,
    )

    effective_batch = trainer.args.per_device_train_batch_size * trainer.args.gradient_accumulation_steps
    steps_per_epoch = max(1, math.ceil(len(dataset) / max(effective_batch, 1)))
    total_steps = int(steps_per_epoch * trainer.args.num_train_epochs)
    print(
        f"Planned: {len(dataset)} samples | effective_batch={effective_batch} | "
        f"steps/epoch≈{steps_per_epoch} | total_steps≈{total_steps} | "
        f"log/eval/save every {SAVE_STEPS} steps",
        flush=True,
    )

    result = trainer.train()
    print("Training metrics:", result.metrics)
    save_path = Path(variant["save_path"])
    save_path.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(save_path, safe_serialization=True)
    tokenizer.save_pretrained(save_path)
    print(f"Saved adapter and tokenizer -> {save_path}")

    del trainer, model, tokenizer
    clear_gpu()
    return str(save_path)


In [ ]:
if RUN_TRAINING:
    variant_map = {variant["name"]: variant for variant in ADAPTER_VARIANTS}
    trained_paths = {}
    for method in TRAIN_METHODS:
        if method not in variant_map:
            raise ValueError(f"Unknown TRAIN_METHODS item: {method}")
        trained_paths[method] = train_one_variant(
            variant_map[method],
            max_seq_length,
            dataset,
            eval_dataset=eval_dataset,
        )
    print("Training complete:", trained_paths)
else:
    print("RUN_TRAINING=False — skipped.")


## ViNewsQA test evaluation

Inference runs in an isolated subprocess with plain Transformers/PEFT, left padding, and batch size 16.
EM and token F1 use the maximum score over every SQuAD gold answer.


In [ ]:
PREFIX_RE = re.compile(r"^(đáp án|answer|câu trả lời)\\s*[:\\-]?\\s*", re.IGNORECASE)


def normalize_text(text):
    text = unicodedata.normalize("NFC", text or "")
    return " ".join(text.lower().translate(str.maketrans("", "", string.punctuation)).split())


def compute_em(prediction, truth):
    return int(normalize_text(prediction) == normalize_text(truth))


def compute_f1(prediction, truth):
    pred_tokens = normalize_text(prediction).split()
    truth_tokens = normalize_text(truth).split()
    if not pred_tokens and not truth_tokens:
        return 1.0
    if not pred_tokens or not truth_tokens:
        return 0.0
    overlap = sum((Counter(pred_tokens) & Counter(truth_tokens)).values())
    if not overlap:
        return 0.0
    precision = overlap / len(pred_tokens)
    recall = overlap / len(truth_tokens)
    return 2 * precision * recall / (precision + recall)


def score_prediction(prediction, gold_answers):
    return (
        max(compute_em(prediction, gold) for gold in gold_answers),
        max(compute_f1(prediction, gold) for gold in gold_answers),
    )


_EVAL_INFER_SUBPROCESS_SRC = r"""from __future__ import annotations
import argparse
import json
import os
import re
import sys
import time
from pathlib import Path

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
SYSTEM_PROMPT = (
    "Bạn là hệ thống hỏi-đáp trích xuất tiếng Việt. Chỉ trả lời bằng một cụm từ xuất hiện "
    "nguyên văn trong đoạn văn, không giải thích và không thêm tiền tố.\\n\\n"
    "Đoạn văn:\\n{context}"
)
PREFIX_RE = re.compile(r"^(đáp án|answer|câu trả lời)\\s*[:\\-]?\\s*", re.IGNORECASE)


def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--adapter-dir", required=True)
    parser.add_argument("--samples-json", required=True)
    parser.add_argument("--output", required=True)
    parser.add_argument("--base-model", required=True)
    parser.add_argument("--max-seq-length", type=int, required=True)
    parser.add_argument("--max-new-tokens", type=int, default=64)
    parser.add_argument("--batch-size", type=int, default=16)
    parser.add_argument("--log-every", type=int, default=100)
    return parser.parse_args()


def clean_prediction(value):
    value = value.strip().split("\\n")[0].strip().strip("\\\"'")
    return PREFIX_RE.sub("", value).strip()


def main():
    if "unsloth" in sys.modules:
        raise RuntimeError("Subprocess isolation failed: unsloth was imported.")
    args = parse_args()
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
    from peft import PeftModel

    adapter_dir = Path(args.adapter_dir)
    adapter_cfg = json.load(open(adapter_dir / "adapter_config.json", encoding="utf-8"))
    peft_type = str(adapter_cfg.get("peft_type") or "").upper()
    base_model = adapter_cfg.get("base_model_name_or_path") or args.base_model
    if str(base_model).lower().startswith("unsloth/"):
        base_model = args.base_model
    full_precision = peft_type == "DELORA"
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    load_kwargs = {"torch_dtype": dtype}
    if not full_precision:
        load_kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=dtype,
        )
        load_kwargs["device_map"] = {"": 0}

    tokenizer = AutoTokenizer.from_pretrained(base_model)
    model = AutoModelForCausalLM.from_pretrained(base_model, **load_kwargs)
    if full_precision:
        model = model.to("cuda")
    model = PeftModel.from_pretrained(model, str(adapter_dir), is_trainable=False)
    model.eval()
    model.config.use_cache = True
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    samples = json.load(open(args.samples_json, encoding="utf-8"))
    prompts = [
        tokenizer.apply_chat_template(
            [
                {"role": "system", "content": SYSTEM_PROMPT.format(context=row["context"])},
                {"role": "user", "content": row["question"]},
            ],
            tokenize=False,
            add_generation_prompt=True,
        )
        for row in samples
    ]
    order = sorted(range(len(samples)), key=lambda index: len(prompts[index]))
    predictions = [None] * len(samples)
    started = time.time()
    done = 0
    for start in range(0, len(order), args.batch_size):
        indices = order[start:start + args.batch_size]
        encoded = tokenizer(
            [prompts[index] for index in indices],
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=args.max_seq_length,
        ).to(model.device)
        with torch.no_grad():
            generated = model.generate(
                **encoded,
                max_new_tokens=args.max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )
        new_tokens = generated[:, encoded["input_ids"].shape[1]:]
        for batch_index, sample_index in enumerate(indices):
            row = samples[sample_index]
            raw = tokenizer.decode(new_tokens[batch_index], skip_special_tokens=True)
            predictions[sample_index] = {
                "id": row["id"],
                "question": row["question"],
                "ground_truth": row["answer"],
                "gold_answers": row["gold_answers"],
                "prediction_raw": raw,
                "prediction": clean_prediction(raw),
            }
        done += len(indices)
        if done == len(samples) or done % max(args.log_every, args.batch_size) < args.batch_size:
            rate = done / max(time.time() - started, 1e-3)
            print(f"[infer] {done}/{len(samples)} | {rate:.2f} samples/s", flush=True)

    with open(args.output, "w", encoding="utf-8") as handle:
        json.dump(predictions, handle, ensure_ascii=False)


if __name__ == "__main__":
    main()
"""


def ensure_eval_infer_script():
    path = Path("eval_infer_subprocess.py")
    if not path.exists() or path.read_text(encoding="utf-8") != _EVAL_INFER_SUBPROCESS_SRC:
        path.write_text(_EVAL_INFER_SUBPROCESS_SRC, encoding="utf-8")
    return path


In [ ]:
def infer_predictions_subprocess(method_name, adapter_path, samples, max_seq_length, log_every):
    import shutil
    import subprocess
    import sys
    import tempfile

    adapter_path = Path(adapter_path)
    if not adapter_path.exists():
        print(f"SKIP {method_name}: missing {adapter_path}")
        return None
    if not (adapter_path / "adapter_model.safetensors").exists():
        raise FileNotFoundError(adapter_path / "adapter_model.safetensors")

    script = ensure_eval_infer_script()
    temp_dir = Path(tempfile.mkdtemp(prefix="vinewsqa_eval_"))
    samples_path = temp_dir / "samples.json"
    output_path = temp_dir / "predictions.json"
    with open(samples_path, "w", encoding="utf-8") as handle:
        json.dump(samples, handle, ensure_ascii=False)
    command = [
        sys.executable, str(script),
        "--adapter-dir", str(adapter_path),
        "--samples-json", str(samples_path),
        "--output", str(output_path),
        "--base-model", BASE_MODEL_NAME,
        "--max-seq-length", str(max_seq_length),
        "--max-new-tokens", str(MAX_NEW_TOKENS),
        "--batch-size", str(INFER_BATCH_SIZE),
        "--log-every", str(log_every),
    ]
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line.rstrip(), flush=True)
    return_code = process.wait()
    if return_code:
        shutil.rmtree(temp_dir, ignore_errors=True)
        raise RuntimeError(f"{method_name} inference failed with exit code {return_code}.")
    with open(output_path, encoding="utf-8") as handle:
        predictions = json.load(handle)
    shutil.rmtree(temp_dir, ignore_errors=True)
    return predictions


def eval_one_adapter(method_name, adapter_path, samples, max_seq_length):
    selected = samples[:EVAL_MAX_SAMPLES] if EVAL_MAX_SAMPLES else samples
    predictions = infer_predictions_subprocess(
        method_name, adapter_path, selected, max_seq_length, EVAL_LOG_EVERY
    )
    if predictions is None:
        return None
    em_values, f1_values = [], []
    for row in predictions:
        em, f1 = score_prediction(row["prediction"], row["gold_answers"])
        row["em"], row["f1"] = em, f1
        em_values.append(em)
        f1_values.append(f1)
    metrics = {
        "method": method_name,
        "adapter": adapter_path,
        "exact_match": round(100 * sum(em_values) / len(em_values), 4),
        "f1": round(100 * sum(f1_values) / len(f1_values), 4),
        "n_samples": len(predictions),
    }
    return {"metrics": metrics, "predictions": predictions}


def export_submission_one_adapter(method_name, adapter_path, samples, max_seq_length):
    selected = samples[:SUBMISSION_MAX_SAMPLES] if SUBMISSION_MAX_SAMPLES else samples
    predictions = infer_predictions_subprocess(
        method_name, adapter_path, selected, max_seq_length, SUBMISSION_LOG_EVERY
    )
    if predictions is None:
        return None
    output_dir = Path(adapter_path)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / "results.json"
    with open(output_path, "w", encoding="utf-8") as handle:
        json.dump({row["id"]: row["prediction"] for row in predictions}, handle, ensure_ascii=False, indent=2)
    print(f"Saved {output_path} ({len(predictions)} predictions)")
    return output_path


In [ ]:
variant_map = {variant["name"]: variant for variant in ADAPTER_VARIANTS}

if RUN_METRIC_EVAL:
    all_results = {}
    for method in EVAL_METHODS:
        result = eval_one_adapter(method, variant_map[method]["save_path"], eval_samples, max_seq_length)
        if result is not None:
            all_results[method] = result

    print("\n" + "=" * 62)
    print(f"ViNewsQA SQuAD 1.1 comparison — {EVAL_SPLIT} ({len(eval_samples)} samples)")
    print(f"{'Method':<12} {'EM':>12} {'F1':>12} {'Samples':>12}")
    print("-" * 62)
    for method, result in all_results.items():
        metrics = result["metrics"]
        print(f"{method:<12} {metrics['exact_match']:>11.2f}% {metrics['f1']:>11.2f}% {metrics['n_samples']:>12}")
    print("=" * 62)

    comparison = {
        "dataset": "ViNewsQA",
        "format": "SQuAD 1.1 extractive QA",
        "eval_split": EVAL_SPLIT,
        "expected_test_size": EXPECTED_TEST_SIZE,
        "base_model": BASE_MODEL_NAME,
        "max_seq_length": max_seq_length,
        "summary": {name: value["metrics"] for name, value in all_results.items()},
        "predictions": {name: value["predictions"] for name, value in all_results.items()},
    }
    with open(COMPARE_EVAL_PATH, "w", encoding="utf-8") as handle:
        json.dump(comparison, handle, ensure_ascii=False, indent=2)
    print("Saved comparison ->", COMPARE_EVAL_PATH)
else:
    print("RUN_METRIC_EVAL=False — skipped.")

if RUN_SUBMISSION_EXPORT:
    for method in EVAL_METHODS:
        export_submission_one_adapter(
            method, variant_map[method]["save_path"], test_samples, max_seq_length
        )
else:
    print("RUN_SUBMISSION_EXPORT=False — skipped.")
